# High-D Blind Likelihood Tilt GPU Experiment

This notebook runs the GPU/PyTorch port of the lightweight high-dimensional closed-form toy.

It tests first-order noisy-coordinate likelihood tilt only:

- `scheduled_tilt`
- `blind_mle_tilt`
- `raw_pnp_tuned`
- `unscaled_noisy`

It intentionally does **not** run the finite Kalman/covariance-inverse method.

## 1. Runtime Setup

In Colab, set `Runtime -> Change runtime type -> GPU` before running. If this notebook is already inside the cloned repo, leave `REPO_URL` empty. Otherwise set it to your GitHub repo URL.

In [ ]:
from pathlib import Path
import os, sys

REPO_URL = ""  # Example: "https://github.com/<user>/<repo>.git"

if not Path("toy_blind_splitting").exists():
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL to your GitHub repo URL, or upload/run this notebook from the repo root.")
    !git clone {REPO_URL} blind_diffusion
    os.chdir("blind_diffusion")

sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

In [ ]:
!pip -q install -r toy_blind_splitting/requirements-gpu.txt

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No CUDA GPU visible. The notebook will run, but it will be slow.")

## 2. Import GPU Runner

In [ ]:
from toy_blind_splitting.src.experiments_highd_likelihood_tilt_torch import run_experiment
from pathlib import Path
import numpy as np

BASE_OUT = Path("toy_blind_splitting/results/colab_highd_likelihood_tilt_gpu")
BASE_OUT.mkdir(parents=True, exist_ok=True)

## 3. Smoke Test

Run this first. It should finish quickly on GPU.

In [ ]:
smoke = run_experiment(
    out=BASE_OUT / "smoke",
    d_values=[50],
    eta_values=[1e-3],
    n_trials=8,
    batch_size=8,
    n_steps=10,
    components=8,
    grid_size=21,
    device_name="auto",
)

## 4. Main GPU Run

Start with this moderate GPU run. Increase `n_trials` after confirming runtime and memory.

Suggested first pass:

- `d = [200, 500, 1000, 2000]`
- `eta = [1e-3, 3e-3, 1e-2]`
- `n_trials = 64`
- `batch_size = 16` or `32`, depending on GPU memory
- `n_steps = 80`

In [ ]:
payload = run_experiment(
    out=BASE_OUT / "main",
    d_values=[200, 500, 1000, 2000],
    eta_values=[1e-3, 3e-3, 1e-2],
    n_trials=64,
    batch_size=16,
    n_steps=80,
    components=16,
    grid_size=41,
    measurement_ratio=0.5,
    methods=["scheduled_tilt", "blind_mle_tilt", "raw_pnp_tuned", "unscaled_noisy"],
    device_name="auto",
    dtype_name="float32",
)

## 5. Summarize Results

In [ ]:
result_path = BASE_OUT / "main" / "data" / "highd_likelihood_tilt_gpu.npz"
data = np.load(result_path, allow_pickle=True)
metrics = data["metrics"]
d_values = data["d_values"]
eta_values = data["eta_values"]
methods = [str(x) for x in data["method_names"]]
metric_names = [str(x) for x in data["metric_names"]]
mse_idx = metric_names.index("mse_true")
meas_idx = metric_names.index("measurement_mse")

print("Aggregate median MSE:")
agg = np.nanmedian(metrics[..., mse_idx], axis=(0, 1, 2))
for i in np.argsort(agg):
    print(f"  {methods[i]:18s} {agg[i]:.4e}")

print("\nBy dimension:")
for di, d in enumerate(d_values):
    vals = np.nanmedian(metrics[di, ..., mse_idx], axis=(0, 1))
    print(f"d={int(d)}")
    for i in np.argsort(vals):
        meas = np.nanmedian(metrics[di, :, :, i, meas_idx])
        print(f"  {methods[i]:18s} mse={vals[i]:.4e} meas={meas:.4e}")

In [ ]:
import pandas as pd
data_dir = BASE_OUT / "main" / "data"
fig_dir = BASE_OUT / "main" / "figures"

print("Generated data files:")
for p in sorted(data_dir.glob("*")):
    print(" ", p)

print("\nGenerated figures:")
for p in sorted(fig_dir.glob("*.png")):
    print(" ", p)

summary = pd.read_csv(data_dir / "summary_by_condition.csv")
summary.head(12)

In [ ]:
from IPython.display import Image, display
for name in [
    "mse_by_dimension.png",
    "mse_by_eta.png",
    "measurement_mse_by_dimension.png",
    "median_sigma_by_dimension.png",
    "sigma_boundary_hits.png",
]:
    display(Image(filename=str(fig_dir / name)))

In [ ]:
from IPython.display import Markdown, display
report_path = BASE_OUT / "main" / "data" / "highd_likelihood_tilt_gpu_report.md"
display(Markdown(report_path.read_text()))

## 6. Optional: Copy Results To Google Drive

Uncomment this if you want persistent storage in Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/blind_diffusion_results
# !cp -r {BASE_OUT} /content/drive/MyDrive/blind_diffusion_results/